# DL vertex finding network training

This notebook is designed to take the input and truth images generated by the <code>make_images.ipynb</code> notebook and train networks for vertex finding. This notebook generates models for each of the U, V and W views for each of the required passes.
    
Most of the cells below will not need any editing, but towards the bottom of the notebook you will find some additional markdown that describes what you may need to edit (essentially just some file locations).

In [1]:
# Automatically reload external libraries that change
%reload_ext autoreload
%autoreload 2

# If a matplotlib plot command is issued, display the results in the notebook
%matplotlib inline

In [2]:
# imaging.py

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from tqdm.notebook import tqdm


def imagify(input, pred, truth, n=3, randomize=True, null_code=0):
    """Process input, prediction and mask data ready for display
    
    Args:
        inputs: Input tensor from a batch
        predictions: Predictions tensor from a batch
        truth: Truth tensor from a batch
        n: The number of images to extract from the batch (default: 3)
        randomize: Choose random images from the batch if True, choose the first n otherwise (default: True)
        null_code: The null mask code (default: 0)
            
    Returns:
        A tuple (if n == 1) or zip of the processed images ready for display.
    """
    # Select the images to process
    choices = np.random.choice(np.array(range(inputs.shape[0])), size=n) if randomize else np.array(range(n))
    input_imgs = input[choices,0,...]
    truth_imgs = truth[choices,...]

    input_imgs = input_imgs.detach().cpu()
    truth_imgs = truth_imgs.detach().cpu()
    pred_imgs = pred[choices,...].detach().cpu()

    # Remove non-hit regions
    mask = truth_imgs == null_code
    pred_imgs = np.argmax(pred_imgs, axis=1)
    pred_imgs = np.ma.array(pred_imgs, mask = mask).filled(0)

    return zip(input_imgs, truth_imgs, pred_imgs) if n > 1 else (input_imgs, truth_imgs, pred_imgs)


def show_batch(epoch, batch, input, pred, truth, null_code=0, n=3, randomize=True):
    """Display the images for a given epoch and batch. Each row is a triplet of input, prediction and mask.

    Args:
        epoch: The current training epoch
        batch: The current training batch
        input: Input tensor from a batch
        pred: Predictions tensor from a batch
        truth: Truth tensor from a batch
        n: The number of images to extract from the batch (default: 3)
        randomize: Choose random images from the batch if True, choose the first n otherwise (default: True)
        null_code: The null mask code (default: 0).
    """
    global vertex_pass, view
    ax = None
    rows, cols, size = 1, 2, 9
    cmap = "magma" #ListedColormap(['black', 'red'])
    bounds = np.linspace(0, 19, 19)
    norm = BoundaryNorm(boundaries=bounds, ncolors=19)
    xtr = dict(cmap=cmap, norm=norm)
    #norm = BoundaryNorm([0., 0.05, 1.], cmap.N)
    #cmap = ListedColormap(['black', 'red', 'yellow'])
    #norm = BoundaryNorm([0., 0.5, 1.5, 2.], cmap.N)
    #xtr = dict(cmap=cmap, norm=norm, alpha=0.7)

    images = imagify(input, pred, truth, n, randomize, null_code)

    for i, imgs in enumerate(images):
        raw, cls, net = imgs
        pair = (cls, net)
        fig, axs = plt.subplots(1, cols, figsize=(cols * size, size))
        for img, ax in zip(pair, axs):
            #ax.imshow(raw, cmap="gist_gray")
            ax.imshow(img, **xtr)
            #ax.imshow(img, cmap="magma")
            ax.axis('off')
        plt.tight_layout()
        save_figure(plt, f"outputs/images/pass{vertex_pass}/{view}/output_{epoch}_{batch}_{i}")
        plt.close(fig)


def save_figure(fig, name):
    """Output a matplotlib figure PNG, PDF and EPS formats.

    Args:
        fig (Figure): The matplotlib figure to save.
        name (str): The output filename excluding extension.
    """
    fig.savefig(name + ".png", facecolor='w')
    fig.savefig(name + ".pdf")
    #fig.savefig(name + ".eps")


def get_supported_formats():
    """Retrieve the supported image formats.

    Returns:
        A dictionary containing strings of file format descriptions keyed by extension.
    """
    return plt.gcf().canvas.get_supported_filetypes()

In [3]:
# analysis.py
from functools import partial

def flatten_model(module):
    children = list(module.children())
    if len(children) == 0:
        return [module]
    else:
        flat_model = []
        for child in children:
            flat_model += flatten_model(child)
        return flat_model


class Hook:
    def __init__(self, id, module, func):
        self.id = id
        self.name = module.__class__.__name__
        self.hook = module.register_forward_hook(partial(func, self))
    
    def remove(self):
        self.hook.remove()
    
    def __del__(self):
        self.remove()


def append_stats(hook, module, input, output):
    if not module.training:
        return
    if not hasattr(hook, 'stats'):
        hook.stats = ([],[],[])
    means, stds, hists = hook.stats
    means.append(output.data.mean())
    stds.append(output.data.std())
    hists.append(output.data.histc(40, -5, 5))

In [4]:
# model.py

import torch.nn as nn
import torch


def maxpool():
    """Return a max pooling layer.
    
        The maxpooling layer has a kernel size of 2, a stride of 2 and no padding.

        Returns:
            The max pooling layer
    """
    return nn.MaxPool2d(kernel_size = 2, stride = 2, padding = 0)


def dropout(prob):
    """Return a dropout layer.

        Args:
            prob: The probability that drop out will be applied.

        Returns:
            The dropout layer
    """
    return nn.Dropout(prob)


def reinit_layer(layer, leak = 0.0, use_kaiming_normal=True):
    """Reinitialises convolutional layer weights.
    
        The default Kaiming initialisation in PyTorch is not optimal, this method
        reinitialises the layers using better parameters

        Args:
            seq_block: The layer to be reinitialised.
            leak: The leakiness of ReLU (default: 0.0)
            use_kaiming_normal: Use Kaiming normal if True, Kaiming uniform otherwise (default: True)
    """
    if isinstance(layer, nn.Conv2d) or isinstance(layer, nn.ConvTranspose2d):
        if use_kaiming_normal:
            nn.init.kaiming_normal_(layer.weight, a = leak)
        else:
            nn.init.kaiming_uniform_(layer.weight, a = leak)
            layer.bias.data.zero_()


class ConvBlock(nn.Module):
    """A convolution block
    """
    
    # Sigmoid activation suitable for binary cross-entropy
    def __init__(self, c_in, c_out, k_size = 3, k_pad = 1):
        """Constructor.

            Args:
                c_in: The number of input channels
                c_out: The number of output channels
                k_size: The size of the convolution filter
                k_pad: The amount of padding around the images
        """
        super(ConvBlock, self).__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, kernel_size = k_size, padding = k_pad, stride = 1)
        self.norm1 = nn.GroupNorm(8, c_out)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(c_out, c_out, kernel_size = k_size, padding = k_pad, stride = 1)
        self.norm2 = nn.GroupNorm(8, c_out)
        self.identity = nn.Conv2d(c_in, c_out, kernel_size = 1, padding = 0, stride = 1)
        reinit_layer(self.conv1)
        reinit_layer(self.conv2)

    def forward(self, x):
        """Forward pass.
        
            Args:
                x: The input to the layer
                
            Returns:
                The output from the layer
        """
        identity = self.identity(x)
        x = self.conv1(x)
        x = self.norm1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.norm2(x)
        return self.relu(x + identity)


class TransposeConvBlock(nn.Module):
    """A tranpose convolution block
    """
    
    def __init__(self, c_in, c_out, k_size = 3, k_pad = 1):
        """Constructor.

            Args:
                c_in: The number of input channels
                c_out: The number of output channels
                k_size: The size of the convolution filter
                k_pad: The amount of padding around the images
        """
        super(TransposeConvBlock, self).__init__()
        self.block = nn.Sequential(
            nn.ConvTranspose2d(c_in, c_out, kernel_size = k_size, padding = k_pad, output_padding = 1, stride = 2),
            nn.GroupNorm(8, c_out),
            nn.ReLU(inplace=True))
        reinit_layer(self.block[0])

    def forward(self, x):
        """Forward pass.
        
            Args:
                x: The input to the layer
                
            Returns:
                The output from the layer
        """
        return self.block(x)

class Sigmoid(nn.Module):
    """A sigmoid activation function that supports categorical cross-entropy
    """
    
    def __init__(self, out_range = None):
        """Constructor.

            Args:
                out_range: A tuple covering the minimum and maximum values to map to
        """
        super(Sigmoid, self).__init__()
        if out_range is not None:
            self.low, self.high = out_range
            self.range = self.high - self.low
        else:
            self.low = None
            self.high = None
            self.range = None
    
    def forward(self, x):
        """Applies the sigmoid function.
        
            Rescales to the specified range if provided during construction
        
            Args:
                x: The input to the layer
                
            Returns:
                The (potentially scaled) sigmoid of the input
        """
        if self.low is not None:
            return torch.sigmoid(x) * (self.range) + self.low
        else:
            return torch.sigmoid(x)

class UNet(nn.Module):
    """A U-Net for semantic segmentation.
    """
    
    def __init__(self, in_dim, n_classes, depth = 4, n_filters = 16, drop_prob = 0.1, y_range = None):
        """Constructor.

            Args:
                in_dim: The number of input channels
                n_classes: The number of classes
                depth: The number of convolution blocks in the downsampling and upsampling arms of the U (default: 4)
                n_filters: The number of filters in the first layer (doubles for each downsample) (default: 16)
                drop_prob: The dropout probability for each layer (default: 0.1)
                y_range: The range of values (low, high) to map to in the output (default: None)
        """
        super(UNet, self).__init__()
        # Contracting Path
        self.ds_conv_1 = ConvBlock(in_dim, n_filters)
        self.ds_conv_2 = ConvBlock(n_filters, 2 * n_filters)
        self.ds_conv_3 = ConvBlock(2 * n_filters, 4 * n_filters)
        self.ds_conv_4 = ConvBlock(4 * n_filters, 8 * n_filters)

        self.ds_maxpool_1 = maxpool()
        self.ds_maxpool_2 = maxpool()
        self.ds_maxpool_3 = maxpool()
        self.ds_maxpool_4 = maxpool()
        
        self.ds_dropout_1 = dropout(drop_prob)
        self.ds_dropout_2 = dropout(drop_prob)
        self.ds_dropout_3 = dropout(drop_prob)
        self.ds_dropout_4 = dropout(drop_prob)
        
        self.bridge = ConvBlock(8 * n_filters, 16 * n_filters)
        
        # Expansive Path
        self.us_tconv_4 = TransposeConvBlock(16 * n_filters, 8 * n_filters)
        self.us_tconv_3 = TransposeConvBlock(8 * n_filters, 4 * n_filters)
        self.us_tconv_2 = TransposeConvBlock(4 * n_filters, 2 * n_filters)
        self.us_tconv_1 = TransposeConvBlock(2 * n_filters, n_filters)

        self.us_conv_4 = ConvBlock(16 * n_filters, 8 * n_filters)
        self.us_conv_3 = ConvBlock(8 * n_filters, 4 * n_filters)
        self.us_conv_2 = ConvBlock(4 * n_filters, 2 * n_filters)
        self.us_conv_1 = ConvBlock(2 * n_filters, 1 * n_filters)

        self.us_dropout_4 = dropout(drop_prob)
        self.us_dropout_3 = dropout(drop_prob)
        self.us_dropout_2 = dropout(drop_prob)
        self.us_dropout_1 = dropout(drop_prob)

        self.output = nn.Sequential(nn.Conv2d(n_filters, n_classes, 1), Sigmoid(y_range))

    def forward(self, x):
        """Forward pass.
        
            Args:
                x: The input to the layer
                
            Returns:
                The output from the layer
        """
        res = x

        # Downsample
        res = self.ds_conv_1(res); conv_stack_1 = res.clone()
        res = self.ds_maxpool_1(res)
        res = self.ds_dropout_1(res)
        
        res = self.ds_conv_2(res); conv_stack_2 = res.clone()
        res = self.ds_maxpool_2(res)
        res = self.ds_dropout_2(res)
        
        res = self.ds_conv_3(res); conv_stack_3 = res.clone()
        res = self.ds_maxpool_3(res)
        res = self.ds_dropout_3(res)
        
        res = self.ds_conv_4(res); conv_stack_4 = res.clone()
        res = self.ds_maxpool_4(res)
        res = self.ds_dropout_4(res)
        
        # Bridge
        res = self.bridge(res)
        
        # Upsample
        res = self.us_tconv_4(res)
        res = torch.cat([res, conv_stack_4], dim=1)
        res = self.us_dropout_4(res)
        res = self.us_conv_4(res)

        res = self.us_tconv_3(res)
        res = torch.cat([res, conv_stack_3], dim=1)
        res = self.us_dropout_3(res)
        res = self.us_conv_3(res)
        
        res = self.us_tconv_2(res)
        res = torch.cat([res, conv_stack_2], dim=1)
        res = self.us_dropout_2(res)
        res = self.us_conv_2(res)
        
        res = self.us_tconv_1(res)
        res = torch.cat([res, conv_stack_1], dim=1)
        res = self.us_dropout_1(res)
        res = self.us_conv_1(res)
        
        output = self.output(res)

        return output

In [5]:
# network.py

# from model import *

import numpy as np
import torch
import torch.optim as opt


def set_seed(seed):
    """Set the various seeds and flags to ensure deterministic performance
    
        Args:
            seed: The random seed
    """
    torch.backends.cudnn.deterministic = True   # Note, can impede performance
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    torch.manual_seed(seed)


def get_class_weights(stats):
    """Get the weights for each class
    
        Each class has a weight inversely proportional to the number of instances in the training set
    
        Args:
            stats: The number of instances of each class
        
        Returns:
            The weights for each class
    """
    if np.any(stats == 0.):
        print("Found a class that doesn't appear")
        idx = np.where(stats == 0.)
        stats[idx] = 1
        weights = 1. / stats
        weights[idx] = 0
    else:
        weights = 1. / stats
    return [weight / sum(weights) for weight in weights]


def load_model_only(filename, num_classes, device):
    """Load a model

        Args:
            filename: The name of the file with the pretrained model parameters
            num_classes: The number of classes available to predict
            weights: The weights to apply to the classes
            device: The device on which to run

        Returns:
            A tuple composed (in order) of the model, loss function, and optimiser
    """
    model = UNet(1, n_classes = num_classes, depth = 4, n_filters = 16, y_range = (0, num_classes - 1))
    model.load_state_dict(torch.load(filename, map_location=device))
    model.eval()
    return model


def load_model(filename, num_classes, weights, device):
    """Load a model

        Args:
            filename: The name of the file with the pretrained model parameters
            num_classes: The number of classes available to predict
            weights: The weights to apply to the classes
            device: The device on which to run

        Returns:
            A tuple composed (in order) of the model, loss function, and optimiser
    """
    model = UNet(1, n_classes = num_classes, depth = 4, n_filters = 16, y_range = (0, num_classes - 1))
    model.load_state_dict(torch.load(filename, map_location=device))
    model.eval()
    loss_fn = nn.CrossEntropyLoss(torch.as_tensor(weights, device=device, dtype=torch.float))
    optim = opt.Adam(model.parameters())
    return model, loss_fn, optim


def save_model(model, input, filename):
    """Save the model
    
        The model is saved as both a pkl file and a TorchScript pt file, which can be loaded via
            model.load_state_dict(torch.load(PATH))
            model.eval()
        
        Args:
            model: The model to save
            input: An example input to the model
            filename: The output filename, without file extension
    """
    torch.save(model.state_dict(), f"{filename}.pkl")

    
def accuracy(pred, truth, nearby=False):
    """Get the network accuracy
    
        Args:
            pred: The network prediction
            truth: The true class
            nearby: Whether to consider adjacent classes acceptable
        
        Returns:
            The accuracy
    """
    target = truth.squeeze(1)
    pred_cls = pred.argmax(dim=1)
    mask = (target != 0)
    if nearby:
        result = abs(pred_cls[mask] - target[mask]) <= 1
    else:
        result = pred_cls[mask] == target[mask]
    return result.float().mean()


def create_model(num_classes, weights, device):
    """Create the model

        Args:
            num_classes: The number of classes available to predict
            weights: The weights to apply to the classes
            device: The device on which to run

        Returns:
            A tuple composed (in order) of the model, loss function, and optimiser
    """
    model = UNet(1, n_classes = num_classes, depth = 4, n_filters = 16, y_range = (0, num_classes - 1))
    loss_fn = nn.CrossEntropyLoss(torch.as_tensor(weights, device=device, dtype=torch.float))
    optim = opt.Adam(model.parameters())
    return model, loss_fn, optim


In [6]:
# data.py

import os

import h5py
import numpy as np
import torch

from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm


class SegmentationDataset(Dataset):
    """Dataset suitable for segmentation tasks, backed by per-class HDF5 files.

        Each class contributes one HDF5 file (written by preprocess.process_file) holding
        two chunked, resizable datasets, 'hits' and 'truth', chunked at one sample per
        chunk. Because HDF5 chunks are read (and decompressed) independently, a single
        __getitem__ call only touches the one chunk it needs - unlike the previous
        .npz-shard format, there's no need to decompress a whole shard up front, and
        therefore no LRU shard cache or shard-grouped sampler required to make random
        access affordable.

        h5py.File handles are NOT opened in __init__: they aren't safe to share across a
        fork, so with num_workers>0 each DataLoader worker process must open its own
        independent handle. Handles are instead opened lazily on first access and cached
        per-process in self._files. __getstate__ strips any open handles before the
        Dataset is copied into a worker (relevant on the 'spawn' start method, e.g.
        Windows/macOS, where the Dataset is pickled rather than fork-copied).

        By default tensors are built on CPU, meant to be moved to GPU in the training loop
        with `.to(device, non_blocking=True)` (works together with num_workers>0 +
        pin_memory for overlapped I/O). If `device` is set to a CUDA device here instead,
        samples are placed on the GPU inside __getitem__ directly. CUDA tensors cannot
        cross process boundaries, so this only works with num_workers=0 (single-process,
        synchronous loading) - SegmentationBunch enforces this automatically when device
        is set.
    """

    def __init__(self, h5_paths, sample_index, transform=False, device=None):
        """Constructor.

            Args:
                h5_paths: Array of HDF5 file paths, one per class, indexed by file_id.
                sample_index: Array of shape (n_samples, 2) with (file_id, local_idx)
                    pairs, one row per dataset sample.
                transform: Whether or not to apply random flip/transpose augmentation
                    (default: False).
                device: If given, samples are created directly on this device (e.g.
                    torch.device('cuda:0')) instead of staying on CPU. Requires
                    num_workers=0 on the DataLoader (default: None -> CPU tensors).
        """
        self.h5_paths = h5_paths
        self.sample_index = sample_index
        self.transform = transform
        # Always resolve to an explicit torch.device, never a bare None - passing
        # device=None into tensor constructors silently falls back to whatever
        # torch.set_default_device() was last set to elsewhere in the notebook, rather
        # than reliably meaning "CPU".
        self.device = device if device is not None else torch.device('cpu')
        self._files = {}  # populated lazily per-worker process: file_id -> h5py.File

    def __len__(self):
        """Retrieve the number of samples in the dataset.

            Returns:
                The number of samples in the dataset
        """
        return len(self.sample_index)

    def _get_file(self, file_id):
        """Lazily open (and cache, per worker process) the HDF5 file for file_id.

            Args:
                file_id: Index into self.h5_paths

            Returns:
                An open h5py.File for that path
        """
        hf = self._files.get(file_id)
        if hf is None:
            hf = h5py.File(self.h5_paths[file_id], 'r')
            self._files[file_id] = hf
        return hf

    def __getitem__(self, idx):
        """Retrieve a sample from the dataset.

            Args:
                idx: The index of the sample to be retrieved

            Returns:
                A (image, mask) tuple of tensors, on self.device (always CPU unless a
                device was explicitly given to the constructor)
        """
        file_id, local_idx = self.sample_index[idx]
        hf = self._get_file(int(file_id))
        # Each of these reads exactly one HDF5 chunk (one event) off disk/decompresses it -
        # not the surrounding shard/file.
        image = hf['hits'][int(local_idx)]
        mask = hf['truth'][int(local_idx)]

        image = torch.as_tensor(np.expand_dims(image, axis=0), device=self.device, dtype=torch.float)
        mask = torch.as_tensor(mask, device=self.device, dtype=torch.long)

        if self.transform:
            should_hflip = torch.rand(1).item() > 0.5
            should_vflip = torch.rand(1).item() > 0.5
            should_transpose = torch.rand(1).item() > 0.5
            if should_hflip:
                image = torch.flip(image, dims=[-1])
                mask = torch.flip(mask, dims=[-1])
            if should_vflip:
                image = torch.flip(image, dims=[-2])
                mask = torch.flip(mask, dims=[-2])
            if should_transpose:
                image = image.transpose(-2, -1)
                mask = mask.transpose(-2, -1)

        return (image, mask)

    def __getstate__(self):
        """Strip open h5py.File handles before this Dataset is copied into a DataLoader
            worker process. File handles aren't picklable (and aren't safe to share across
            a fork even when pickling is skipped), so each worker must lazily reopen its
            own handles via _get_file on first access instead.

            Returns:
                A dict suitable for pickling/copying, with self._files reset to empty
        """
        state = self.__dict__.copy()
        state['_files'] = {}
        return state


def build_sample_index(h5_paths, counts):
    """Build a (file_id, local_idx) index covering every sample across a set of HDF5 files.

        Args:
            h5_paths: List/array of HDF5 file paths, in file_id order.
            counts: Array of per-file sample counts (h5_paths[i] has counts[i] samples).

        Returns:
            A numpy array of shape (sum(counts), 2) with (file_id, local_idx) pairs.
    """
    total = int(np.sum(counts))
    sample_index = np.empty((total, 2), dtype=np.int64)
    pos = 0
    for file_id, n in enumerate(counts):
        n = int(n)
        sample_index[pos:pos + n, 0] = file_id
        sample_index[pos:pos + n, 1] = np.arange(n)
        pos += n
    return sample_index


def _h5_length(path):
    """Get a HDF5 file's sample count from its dataset shape.

        Args:
            path: Path to a data.h5 file written by preprocess.process_file

        Returns:
            The number of samples ('hits' dataset length) in the file

        Note:
            This just reads a dataset's .shape attribute - no decompression needed, unlike
            the old .npz format where getting a shard's length required decompressing the
            entire array. Cheap enough to call once per file at construction time.
    """
    with h5py.File(path, 'r') as hf:
        return hf['hits'].shape[0]


class SegmentationBunch():
    """Associates batches of training and validation datasets suitable for segmentation
        tasks, reading from the per-class HDF5 files written by preprocess.process_file
        (root_dir/<class_dir>/data.h5 - one file per class, not many shard files).
    """

    def __init__(self, root_dir, balance_map, batch_size, valid_pct=0.1, test_pct=0.0,
                 transform=False, shuffle_training=True, num_workers=8, pin_memory=True, device=None):
        """Constructor.

            Args:
                root_dir: The top-level directory containing per-class subdirectories
                balance_map: Dict mapping the different classes (NuMI/numu, NuMI/nue, etc...)
                    to their directory (relative to root_dir, containing a data.h5 written
                    by preprocess.process_file) and target fraction of the total, e.g.:
                        {
                            'class_1': {'dir': 'NuMI/numu', 'fraction': 0.25},
                            'class_2': {'dir': 'BNB/numu', 'fraction': 0.5},
                            'class_3': {'dir': 'BNB/nue', 'fraction': 0.25}
                        }
                    Fractions must sum to 1.0. Sizes are normalized to the bottleneck class.
                batch_size: The batch size
                valid_pct: The fraction of samples to be used for validation (default: 0.1)
                test_pct: The fraction of samples reserved for testing, currently unused for
                    splitting but kept to preserve the (valid_pct + test_pct) < 1 sanity check
                    (default: 0.0)
                transform: Whether or not to apply augmentation to the training set (default: False)
                num_workers: DataLoader worker processes for parallel I/O (default: 8).
                    Ignored (forced to 0) if device is set - see device below. Each worker
                    opens its own HDF5 file handles lazily (see SegmentationDataset), which
                    is cheap compared to decompressing a whole .npz shard, so parallel
                    workers are the normal, expected configuration here.
                pin_memory: Whether to use pinned host memory so H2D copies can be async
                    (default: True). Ignored (forced to False) if device is set, since pinned
                    memory only matters for CPU tensors being copied to GPU.
                device: If given (e.g. torch.device('cuda:0')), samples are created directly
                    on this device inside the Dataset instead of staying on CPU. CUDA tensors
                    cannot cross process boundaries, so this forces num_workers=0 and
                    pin_memory=False - loading becomes single-process and synchronous, and
                    each __getitem__ call blocks the GPU until the sample is ready. Default:
                    None -> CPU tensors, parallel loading via num_workers.
        """
        assert (valid_pct + test_pct) < 1.
        total_fraction = sum(cfg['fraction'] for cfg in balance_map.values())
        assert np.isclose(total_fraction, 1.0), "Fractions in balance_map must sum to 1.0"

        if device is not None:
            if num_workers > 0:
                print(f"device={device} was set: forcing num_workers=0, shuffle_training=False and pin_memory=False "
                      f"(was {num_workers}) since CUDA tensors can't cross process boundaries")
            num_workers = 0
            shuffle_training = False
            pin_memory = False

        class_names = list(balance_map.keys())
        h5_paths = np.array([
            os.path.join(root_dir, balance_map[c]['dir'], 'data.h5') for c in class_names
        ])
        # Cheap: just reads each file's dataset shape, no decompression.
        counts = np.array([_h5_length(p) for p in tqdm(h5_paths, desc='Reading class sizes')])
        per_class_count = dict(zip(class_names, counts))

        # Find the total dataset capacity dictated by the bottleneck class
        max_total = min(per_class_count[c] / balance_map[c]['fraction'] for c in balance_map)

        train_rows, valid_rows = [], []

        for file_id, class_name in enumerate(tqdm(class_names, desc='Sampling classes')):
            config = balance_map[class_name]
            n_to_sample = int(max_total * config['fraction'])
            n_total = int(counts[file_id])

            chosen_local = np.random.permutation(n_total)[:n_to_sample]
            chosen = np.empty((n_to_sample, 2), dtype=np.int64)
            chosen[:, 0] = file_id
            chosen[:, 1] = chosen_local

            n_valid = int(len(chosen) * valid_pct)
            perm = np.random.permutation(len(chosen))
            valid_rows.append(chosen[perm[:n_valid]])
            train_rows.append(chosen[perm[n_valid:]])

        train_index = np.concatenate(train_rows)
        valid_index = np.concatenate(valid_rows)
        # No manual shuffle needed here (and no shard-grouped sampler, unlike the old
        # .npz-shard version): DataLoader(shuffle=True) below reshuffles every epoch, and
        # because each HDF5 read only costs one chunk regardless of access order, plain
        # random per-sample order no longer thrashes a cache the way it did with shards.

        train_ds = SegmentationDataset(h5_paths, train_index, transform=transform, device=device)
        valid_ds = SegmentationDataset(h5_paths, valid_index, transform=False, device=device)

        self.train_dl = DataLoader(
            train_ds, batch_size=batch_size, shuffle=shuffle_training, drop_last=True,
            num_workers=num_workers, pin_memory=pin_memory,
            persistent_workers=(num_workers > 0), prefetch_factor=4 if num_workers > 0 else None
        )
        self.valid_dl = DataLoader(
            valid_ds, batch_size=batch_size, shuffle=False, drop_last=True,
            num_workers=num_workers, pin_memory=pin_memory,
            persistent_workers=(num_workers > 0), prefetch_factor=4 if num_workers > 0 else None
        )

    def count_classes(self, num_classes, device=None, read_batch_size=20000):
        """Count the number of instances of each class in the training set.
 
            Args:
                num_classes: The number of classes in the training set
                device: Device to accumulate the running count on (default: None -> CPU).
                    Batches arrive on CPU from the DataLoader; pass a CUDA device here only
                    if you want the accumulation itself done on GPU.
                read_batch_size: Max samples to fancy-index out of a file in one read
                    (default: 2000). A single file can contribute hundreds of thousands of
                    samples to the training split - fancy-indexing them all in one h5py call
                    would materialize that entire subset in memory at once (and upcasting to
                    torch.long before bincount made that an extra 8x on top, since truth is
                    stored as uint8). Reading in bounded batches instead keeps peak memory to
                    roughly read_batch_size * image_height * image_width regardless of how
                    large any one file's split is.
 
            Returns:
                A numpy array of the number of instances of each class
        """
        resolved_device = device if device is not None else torch.device('cpu')
        ds = self.train_dl.dataset
 
        # Group this Dataset's training sample indices by which HDF5 file they live in, so
        # each file is opened exactly once here.
        file_to_local = {}
        for file_id, local_idx in ds.sample_index:
            file_to_local.setdefault(int(file_id), []).append(int(local_idx))
 
        count = torch.zeros(num_classes, dtype=torch.long, device=resolved_device)
        for file_id, local_indices in tqdm(file_to_local.items(), desc='Counting classes'):
            # h5py requires fancy-index lists to be strictly increasing.
            local_indices = sorted(local_indices)
            with h5py.File(ds.h5_paths[file_id], 'r') as hf:
                truth_ds = hf['truth']
                for start in tqdm(range(0, len(local_indices), read_batch_size), desc=f'Counting for batch'):
                    batch_idx = local_indices[start:start + read_batch_size]
                    truth = torch.from_numpy(truth_ds[batch_idx])  # stays uint8 here
                    truth = truth.to(resolved_device, non_blocking=True)
                    # .long() only on this bounded batch, not the whole file's selection
                    count += torch.bincount(truth.flatten().long(), minlength=num_classes)
        return count.cpu().numpy()

# Run network training here

The key parameters that will need editing are the view and the pass to be trained. Each view/pass combination has its own network. Views are specified using the standard U, V, W nomenclature (and is consistent with the file naming conventions from previous steps), while the pass is either pass 1 or pass 2.

The respective variables can be set in the cell below via <code>view</code> and <code>vertex_pass</code>.

If you edited the <code>thresholds</code> variable at the <code>make_images</code> stage, you may need to update the <code>NUM_CLASSES</code> variable to reflect any change in the number of thresholds. Note that this value should be equal to the length of the <code>thresholds</code> variable, despite this variable specifying bin edges, because one extra class is required to represent the null case where a pixel has no hits in it.

<code>batch_size</code> can, of course, be varied according to the available resources of your GPU, but as a semantic segmentation network you'll need a lot of memory on your GPU to increase this beyond the current default of 32.

The <code>image_path</code> variable should contain the path to the images generated by the <code>make_images</code> notebook (i.e. <code>global_path</code>), and will expect to find the <code>Hits</code> and <code>Truth</code> folders within that path).

> Note that the code has been updated to allow for sample balancing: taking as input the `balance_map` dictionary, it can, for each sample class, consider a different fraction.
> The dict. is required to follow this schema
> ```python
> {
>   'class_1': {'dir': 'path_1', 'fraction': 0.75},
>   'class_2': {'dir': 'path_2', 'fraction': 0.25}
> }
> ```

Note that the cells below will count the class representation in the training set, determine how to weight them and print this out. It's worth taking a look at this output to ensure all classes are represented, as training will fail if they are not.

You will want to set the number of epochs, <code>n_epochs</code> to train for. This is not easy to determine a priori, but 20 is a reasonable starting point (plots of the loss function and accuracy are produced to help you determine when the network has effectively trained).

<code>model_name</code> acts as a prefix for saving the model. The state of the model is saved after every epoch, with a suffix indicating the epoch number.

Once you are happy with the variable values you can run all of the cells in this section in order (having run all of the cells above), with the final cell in this section actually performing the training.

# First training: perfectly balanced 50-50/50-50 samples

In [8]:
# This line is important for GPU running, otherwise some weights end up on the CPU
torch.set_default_tensor_type(torch.cuda.FloatTensor)

view = "W"
vertex_pass = 1
the_seed = 42
gpu = torch.device('cuda:0')
batch_size=150
NUM_CLASSES = 20   # NULL = 0, various distance bands 1-19
# image_path = f"Accel/Pass{vertex_pass}/Images{view}"
# image_path = '/exp/icarus/data/users/msotgia/vertexStudies/forTraining/ICARUS_DlVertex'
image_path = '/home/msotgia/vertexOnEaf/ICARUS_DlVertex_HDF5' # For processing on EAF w/o overhead

balance_map = {
    'numi_numu': {'dir': f'NuMI/numu/Pass{vertex_pass}/Images{view}', 'fraction': 0.25},
    'numi_nue':  {'dir': f'NuMI/nue/Pass{vertex_pass}/Images{view}',  'fraction': 0.25},
    'bnb_numu':  {'dir': f'BNB/numu/Pass{vertex_pass}/Images{view}',  'fraction': 0.25},
    'bnb_nue':   {'dir': f'BNB/nue/Pass{vertex_pass}/Images{view}',   'fraction': 0.25}
}

n_epochs = 20
model_name = "icarus_pass1_fully_balanced_beam_flavour"

for subdir in ["models", "stats", "images"]:
    dir = f"outputs/{subdir}/pass{vertex_pass}/{view}"
    if not os.path.exists(dir):
        os.makedirs(dir)

In [9]:
# main.py

#from data import *
#from network import *

set_seed(the_seed)
bunch = SegmentationBunch(image_path, balance_map, batch_size=batch_size, valid_pct = 0.25, device=gpu)
train_stats = bunch.count_classes(NUM_CLASSES)
weights = get_class_weights(train_stats)

device=cuda:0 was set: forcing num_workers=0, shuffle_training=False and pin_memory=False (was 8) since CUDA tensors can't cross process boundaries


Reading class sizes:   0%|          | 0/4 [00:00<?, ?it/s]

Sampling classes:   0%|          | 0/4 [00:00<?, ?it/s]

Counting classes:   0%|          | 0/4 [00:00<?, ?it/s]

Counting for batch:   0%|          | 0/5 [00:00<?, ?it/s]

Counting for batch:   0%|          | 0/5 [00:00<?, ?it/s]

Counting for batch:   0%|          | 0/5 [00:00<?, ?it/s]

Counting for batch:   0%|          | 0/5 [00:00<?, ?it/s]

`weights` before the changes were

```python
[np.float64(8.623750719476234e-08),
 np.float64(0.007172987689688895),
 np.float64(0.0007584045062301771),
 np.float64(0.00037038372234998225),
 np.float64(0.00024288647877137123),
 np.float64(0.000198978541093642),
 np.float64(0.00019057576343509776),
 np.float64(0.0001858506259661026),
 np.float64(0.0003745255916270518),
 np.float64(0.000288019790190076),
 np.float64(0.00048425659795097424),
 np.float64(0.0007499850008340051),
 np.float64(0.001114525094378949),
 np.float64(0.0016114242089710934),
 np.float64(0.0016443771616277915),
 np.float64(0.002822851947461105),
 np.float64(0.004856441516708472),
 np.float64(0.011101357805652648),
 np.float64(0.08215330262912612),
 np.float64(0.8836787790904292)]
 ```

 Below is after 

In [10]:
weights

[np.float64(8.595516395945997e-08),
 np.float64(0.007149999886758128),
 np.float64(0.0007567358460601457),
 np.float64(0.00036953976715953436),
 np.float64(0.00024207916886665677),
 np.float64(0.0001981138621505619),
 np.float64(0.00018945002017485001),
 np.float64(0.00018460452360552428),
 np.float64(0.00037189264827564483),
 np.float64(0.00028662952400715743),
 np.float64(0.0004818363438897145),
 np.float64(0.0007500658953065594),
 np.float64(0.0011114845160855623),
 np.float64(0.0015977414403884741),
 np.float64(0.0016308937968408056),
 np.float64(0.00281466752813729),
 np.float64(0.004839165842035388),
 np.float64(0.011074782014265799),
 np.float64(0.08162353038723805),
 np.float64(0.8843267010335901)]

In [11]:
np.savez(f'outputs/stats/pass{vertex_pass}/{view}/weights_{model_name}.npz', weights)

In [12]:
train_losses = torch.zeros(n_epochs * len(bunch.train_dl), device=gpu)
val_losses = torch.zeros(n_epochs, device=gpu)
batch_losses = torch.zeros(len(bunch.valid_dl), device=gpu)

train_accs = torch.zeros(n_epochs * len(bunch.train_dl), device=gpu)
val_accs = torch.zeros(n_epochs, device=gpu)
batch_accs = torch.zeros(len(bunch.valid_dl), device=gpu)

In [13]:
# Standard model creation
model, loss_fn, optim = create_model(NUM_CLASSES, weights, gpu)

i = 0
start = 0
finish = n_epochs

In [ ]:
from tqdm.notebook import tqdm
set_seed(the_seed)
for e in tqdm(range(start, finish), desc='Training'):
    model = model.train()
    n_batches = len(bunch.train_dl)
    for b, batch in enumerate(tqdm(bunch.train_dl, desc=f'For epoch {e}, training over batches')):
        x, y = batch
        pred = model.forward(x)
        loss = loss_fn(pred, y)

        train_losses[i] = loss.item()
        train_accs[i] = accuracy(pred, y, nearby=False)

        loss.backward()
        optim.step()
        #scheduler.step()
        optim.zero_grad()
        i += 1
        if b == (n_batches - 1):
            save_model(model, x, f"outputs/models/pass{vertex_pass}/{view}/{model_name}_{e}")

    # Validate
    model = model.eval()
    with torch.no_grad():
        for b, batch in enumerate(tqdm(bunch.valid_dl, desc=f'For epoch {e}, validating over batches')):
            x, y = batch
            pred = model.forward(x)
            loss = loss_fn(pred, y)
            
            batch_losses[b] = loss.item()
            batch_accs[b] = accuracy(pred, y, nearby=False)
        val_losses[e] = torch.mean(batch_losses)
        val_accs[e] = torch.mean(batch_accs)

    np.savez(f'outputs/stats/pass{vertex_pass}/{view}/losses_{model_name}_{e}.npz', 
             train_losses.cpu(), val_losses.cpu(), batch_losses.cpu(), train_accs.cpu(), val_accs.cpu(), batch_accs.cpu())

    

Training:   0%|          | 0/20 [00:00<?, ?it/s]

For epoch 0, training over batches:   0%|          | 0/2387 [00:00<?, ?it/s]

# Assess network performance

Having trained a network, you can look at its performance - this should be run immediately after the network has finished training. The cells below should not require any editing.

The first cell runs over a single batch from the validation set and produuces images allowing you to compare the truth (left image) to the network classification (right image), though it is worth noting that the <code>show_batch</code> function produces all images in the same folder and does not uniquely identify the view or pass, so if you want to retain them, you'll want to move them between runs.

The next three cells produce plots showing the evolution of the network across epochs. Ideally you want to see a plateauing of the loss and accuracy to establish a well trained model, with no evidence that the training and validation performance are diverging (you can always select a model from an earlier epoch before divergence if the network appears to be overfitting - or get more training samples if the network is not adequately trained).

The final cell in this section simply saves the evolution history of the network to allow easy plot regenertion if needed.

In [ ]:
set_seed(the_seed)
model = model.eval()
with torch.no_grad():
    for b, batch in enumerate(bunch.valid_dl):
        x, y = batch
        pred = model.forward(x)
        show_batch(finish, b, x, pred, y, n=32, randomize=False)
        break

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams['lines.linewidth'] = 3
mpl.rcParams['axes.titlesize'] = 32
mpl.rcParams['axes.labelsize'] = 32
mpl.rcParams['xtick.labelsize'] = 22
mpl.rcParams['ytick.labelsize'] = 22
mpl.rcParams['legend.fontsize'] = 24

In [ ]:
fig = plt.figure(figsize=(20, 15))
plt.xlabel('epoch')
plt.ylabel('metric')

tl = torch.mean(train_losses.reshape([n_epochs, -1]), axis=1).detach().cpu()
vl = val_losses.detach().cpu()
plt.plot(tl, label="training loss")
plt.plot(vl, label="validation loss")

plt.legend()

fig.savefig(f"outputs/stats/pass{vertex_pass}/{view}/stats_loss_{vertex_pass}_{view}.pdf", dpi=200)
fig.savefig(f"outputs/stats/pass{vertex_pass}/{view}/stats_loss_{vertex_pass}_{view}.png", dpi=200, facecolor='w')

In [ ]:
fig = plt.figure(figsize=(20, 15))
plt.xlabel('epoch')
plt.ylabel('metric')

ta = torch.mean(train_accs.reshape([n_epochs, -1]), axis=1).detach().cpu()
va = val_accs.detach().cpu()
plt.plot(ta, label="training accuracy")
plt.plot(va, label="validation accuracy")

plt.legend()

fig.savefig(f"outputs/stats/pass{vertex_pass}/{view}/stats_acc_{vertex_pass}_{view}.pdf", dpi=200)
fig.savefig(f"outputs/stats/pass{vertex_pass}/{view}/stats_acc_{vertex_pass}_{view}.png", dpi=200, facecolor='w')

In [ ]:
with open(f'outputs/stats/pass{vertex_pass}/{view}/train_loss_{vertex_pass}_{view}_20.npy', 'wb') as f:
    np.save(f, tl)
with open(f'outputs/stats/pass{vertex_pass}/{view}/val_loss_{vertex_pass}_{view}_20.npy', 'wb') as f:
    np.save(f, vl)
with open(f'outputs/stats/pass{vertex_pass}/{view}/train_accs_{vertex_pass}_{view}_20.npy', 'wb') as f:
    np.save(f, ta)
with open(f'outputs/stats/pass{vertex_pass}/{view}/val_accs_{vertex_pass}_{view}_20.npy', 'wb') as f:
    np.save(f, va)

# Generating a TorchScript network

The network was trained on a GPU, but ultimately runs on a CPU in a C++ context. This means that the network must be converted to TorchScript format. This can be performed using the cell below and can be run at any time - it need not be run immediately after training the network, because it only requires access to a saved model state.

The only parameters requiring editing here are the location of the input file; which is the save model from the chosen training epoch (so some combination of the <code>moidel_name</code> and epoch with a <code>.pkl</code> extension), the <code>output_filename</code>, which should have a <code>.pt</code> extension, and also the number of classes <code>NUM_CLASSES</code>, which should, of course, match the previouslyt specified value.

The resultant <code>.pt</code> files are what will ultimately be loaded by Pandora for network inference.

In [ ]:
# This line is important for ensuring all tensors exist on the same device
torch.set_default_tensor_type(torch.FloatTensor)

filename = f"outputs/models/pass{vertex_pass}/{view}/dunefd_hd_accel_19.pkl"
output_filename = f"PandoraUnet_Vertex_DUNEFD_Accel_{vertex_pass}_{view}.pt"
the_seed = 42
device = torch.device('cpu')
NUM_CLASSES = 20

set_seed(the_seed)

model = load_model_only(filename, NUM_CLASSES, device)

sm = torch.jit.script(model)
sm.save(output_filename)

# Confusion Matrices

In [ ]:
# This line is important for GPU running, otherwise some weights end up on the CPU
torch.set_default_tensor_type(torch.cuda.FloatTensor)

view = "W"
vertex_pass = 1
the_seed = 42
gpu = torch.device('cuda:0')
batch_size=1
NUM_CLASSES = 20   # NULL = 0, various distance bands 1-19
image_path = f"Accel/Pass{vertex_pass}/Images{view}"

for subdir in ["models", "stats", "images"]:
    dir = f"outputs/{subdir}/pass{vertex_pass}/{view}"
    if not os.path.exists(dir):
        os.makedirs(dir)

set_seed(the_seed)
bunch = SegmentationBunch(image_path, "Hits", "Truth", batch_size=batch_size, valid_pct = 0.25, device=gpu)

filename = f"outputs/models/pass{vertex_pass}/{view}/dunefd_hd_accel_19.pkl"
NUM_CLASSES = 20

set_seed(the_seed)

model = load_model_only(filename, NUM_CLASSES, gpu)

In [ ]:
import scipy.stats as stats
binning = np.linspace(0, 20, 21, dtype=int)

model = model.to(gpu)
confusion = np.zeros((20,20))
for img, cls in bunch.valid_dl:
    img = img.to(gpu)
    output = model(img)
    _, preds = torch.max(output, 1)
    
    cls_detached = cls.cpu().numpy().flatten()
    preds_detached = preds.cpu().numpy().flatten()
    
    H, *_ = stats.binned_statistic_2d(preds_detached, cls_detached, None,
                                      bins=[binning, binning], statistic='count')
    confusion += H

In [ ]:
temporary = confusion.copy()
fig = plt.figure(figsize=(15, 10))
plt.xlabel('true class')
plt.ylabel('fraction')
plt.step(list(np.arange(1, 20)), np.sum(temporary[1:], axis=1) / np.sum(temporary[1:]), where="mid")

save_figure(fig, f"outputs/stats/pass{vertex_pass}/{view}/true_class_dist_{vertex_pass}_{view}")

In [ ]:
sums = np.sum(confusion, axis=1).repeat(20).reshape((20,20))
confusion /= sums

In [ ]:
print(f"--- Class Accuracy")
for t in range(confusion.shape[0]):
    print(f"{t:2}: {100*(confusion[t,t] / confusion[t].sum()):.1f}")
print()

In [ ]:
confusion[0,:] = 0
confusion[:,0] = 0

sums = np.sum(confusion, axis=1).repeat(20).reshape((20,20))
sums[0,:] = 1
confusion /= sums

print(f"--- Class Accuracy")
for t in range(confusion.shape[0]):
    print(f"{t:2}: {100*confusion[t,t]:.1f}")
print()

In [ ]:
fig = plt.figure(figsize=(15, 15))
plt.xlabel('truth')
plt.ylabel('network')
plt.imshow(confusion)
plt.colorbar()
save_figure(fig, f"outputs/stats/pass{vertex_pass}/{view}/confusion_{vertex_pass}_{view}")

for t in range(20):
    for n in range(20):
        print(f"{confusion[t, n]:.2f}", end=" ")
    print()

'test'